In [1]:
import time
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, label_binarize 
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix, precision_score, recall_score,ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import cross_validate, StratifiedKFold, LeaveOneOut, cross_val_predict, GridSearchCV
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_selection import SelectFromModel
from itertools import product
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

In [2]:
# make a base folder to store the plots
base_folder = f"plots/"
os.makedirs(base_folder, exist_ok=True)

In [3]:
#random seed
np.random.seed(22)

In [4]:
#read the data 
datax   = pd.read_csv("Train_call.tsv",      sep="\t", header=0)
datay   = pd.read_csv("Train_clinical.tsv",  sep="\t", header=0)

#make into long dataframe
datax_long = pd.melt(
    datax,
    id_vars=["Chromosome","Start","End","Nclone"],    # vaste kolommen
    var_name="Sample",
    value_name="Value"
)

#merge with the classes of the subtypes of cancer
merged = pd.merge(
    datax_long,
    datay,                    
    on="Sample",
    how="inner"            
)

#make unique genoimic segment into one variable
merged["GenomicSegment"] = (
    "Seg_" + merged["Chromosome"].astype(str)
          + "_" + merged["Start"].astype(str)
          + "_" + merged["End"].astype(str)
)

# pivot long into wide format 
wide_data = merged.pivot_table(
    index=["Sample","Subgroup"],
    columns="GenomicSegment",
    values="Value"
).reset_index()


# drop sample and subgroep and make subgroup the target value
X = wide_data.drop(columns=["Sample","Subgroup"])
subgroup_cat = wide_data["Subgroup"].astype("category") # for the code2lable
y = subgroup_cat.cat.codes

In [5]:
class CorrSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.9):
        # Initialize selector with a correlation cutoff threshold
        self.threshold = threshold

    def fit(self, X, y=None):
        # Convert X to numpy array for corrcoef
        X = np.array(X)
        # compute absolute correlation matrix between features
        corr = np.abs(np.corrcoef(X, rowvar=False))
        # prepare a set to hold indices of features to drop
        drop = set()
        n = X.shape[1]  # number of features

        # loop over upper triangle of the correlation matrix
        for i in range(n):
            for j in range(i+1, n):
                # if two features are more correlated than threshold, mark j for dropping
                if corr[i, j] > self.threshold:
                    drop.add(j)

        # keep indices that were not marked for dropping
        self.keep_idx_ = [i for i in range(n) if i not in drop]
        return self

    def transform(self, X):
        # apply selection: return only the columns in keep_idx_
        return np.array(X)[:, self.keep_idx_]


In [6]:
############################################## rf hyperparmter grid

rf_param_grid = {
    'clf__n_estimators':      [50, 100, 200],
    'clf__max_depth':         [None, 5, 10],
    'clf__min_samples_split': [2, 5, 10]
}
############################################# feature selection grid
#rf importance 
rf_top_features  = [50, 100, 150]

#pca variance
pca_variance     = [0.85, 0.90, 0.95]

#correlation treshold
corr_thresholds  = [0.95, 0.90,0.85]

n_repeats       = 5                     # (7) Repeat entire 3-fold procedure this many times
outer_splits    = 3                     # (6) 3 equal parts for outer CV
inner_folds     = 10                    # (1) 10-fold inner CV to tune FS & HP
results = []

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# 7) Outer repetition: repeat the full 3-fold CV n_repeats times
# ─────────────────────────────────────────────────────────────────────────────
for rep in range(n_repeats):
    
    # 6) Outer 3-fold CV to split into three equal parts
    outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=rep)
    
    for outer_fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        # **Use positional indexing** on the DataFrame/Series
        X_train_outer = X.iloc[train_idx]
        X_test_outer  = X.iloc[test_idx]
        y_train_outer = y.iloc[train_idx]
        y_test_outer  = y.iloc[test_idx]
        
        # 1) Inner 10-fold CV on the 2/3 training portion to tune FS & HP
        inner_cv = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=rep)
        
        for fs_method in ['rf_imp', 'pca', 'corr']:
            
            # Build pipeline + parameter grid
            if fs_method == 'rf_imp':
                pipe = Pipeline([
                    ('scale', StandardScaler()),
                    ('sel',   SelectFromModel(RandomForestClassifier(random_state=rep),
                                             threshold=-np.inf)),
                    ('clf',   RandomForestClassifier(random_state=rep))
                ])
                param_grid = {
                    'sel__max_features': rf_top_features,
                    **rf_param_grid
                }
            
            elif fs_method == 'pca':
                pipe = Pipeline([
                    ('scale', StandardScaler()),
                    ('pca',   PCA()),
                    ('clf',   RandomForestClassifier(random_state=rep))
                ])
                param_grid = {
                    'pca__n_components': pca_variance,
                    **rf_param_grid
                }
            
            else: # corr
                pipe = Pipeline([
                    ('scale', StandardScaler()),
                    ('corr',  CorrSelector()), 
                    ('clf',   RandomForestClassifier(random_state=rep))
                ])
                param_grid = {
                    'corr__threshold': corr_thresholds,
                    **rf_param_grid
                }
            
            # 2) Inner CV grid‐search: select best FS param + classifier HP
            grid = GridSearchCV(pipe, param_grid, cv=inner_cv,
                                scoring='accuracy', n_jobs=-1)
            grid.fit(X_train_outer, y_train_outer)
            
            best_model  = grid.best_estimator_
            best_params = grid.best_params_
            
            # Extract chosen FS param and #features
            if fs_method == 'rf_imp':
                fs_param    = best_params['sel__max_features']
                num_features = best_model.named_steps['sel']\
                                          .transform(X_train_outer).shape[1]
            elif fs_method == 'pca':
                fs_param    = best_params['pca__n_components']
                num_features = best_model.named_steps['pca'].n_components_
            else: # corr
                fs_param    = best_params['corr__threshold']
                num_features = best_model.named_steps['corr']\
                                          .transform(X_train_outer).shape[1]
            
            # 3) Train final predictor on full outer-training set
            best_model.fit(X_train_outer, y_train_outer)
            
            # 4) Training performance t_j*: 10-fold CV on the outer-training set
            train_res = cross_validate(
                best_model, X_train_outer, y_train_outer,
                cv=inner_cv,
                scoring={'acc':'accuracy','auc':'roc_auc_ovr','f1':'f1_macro'},
                n_jobs=-1
            )
            train_acc = np.mean(train_res['test_acc'])
            train_auc = np.mean(train_res['test_auc'])
            train_f1  = np.mean(train_res['test_f1'])
            
            # 5) validate on held-out outer test set x_(j)
            y_pred = best_model.predict(X_test_outer)  
            # get probability matrix for multiclass auc
            y_proba = best_model.predict_proba(X_test_outer)

            # compute accuracy on validation fold
            val_acc = accuracy_score(y_test_outer, y_pred)

            # compute multiclass auc on validation fold (one-vs-rest)
            val_auc = roc_auc_score(
                y_test_outer,      # true labels
                y_proba,           # probability array, shape (n_samples, n_classes)
                multi_class='ovr'   # one-vs-rest setting for multiclass
            )

            # compute f1 (macro) on validation fold
            val_f1 = f1_score(y_test_outer, y_pred, average='macro')

            # collect this fold’s results
            results.append({
                'repeat':        rep,           # repeat index (7)
                'outer_fold':    outer_fold,    # fold index (6)
                'fs_method':     fs_method,     # feature-selection method
                'fs_param':      fs_param,      # chosen fs parameter
                'n_features':    num_features,  # number of selected features
                'train_accuracy': train_acc,     # average train accuracy (4)
                'train_auc':     train_auc,     # average train auc (4)
                'train_f1':      train_f1,      # average train f1 (4)
                'val_accuracy':  val_acc,       # validation accuracy (5)
                'val_auc':       val_auc,       # validation auc (5)
                'val_f1':        val_f1         # validation f1 (5)
            })

# Build the final DataFrame
results_df = pd.DataFrame(results)
print(results_df.head())

C:\Users\youpz\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


In [ ]:
#20 results because you only ever save one row per outerfold per FS parameter not every hyperparameter combination.
results_df.sort_values('accuracy', ascending=False).reset_index(drop=True)

In [ ]:
#get index of best row based on auc score 
best_idx_auc = results_df['auc'].idxmax()
best_row_AUC = results_df.loc[best_idx_auc]

#print best row
print("Best settings      *AUC*     \n\n", best_row_AUC,'\n')

#get index of best row based on accuracy score 
best_idx_accuracy = results_df['accuracy'].idxmax()
best_row_accuracy = results_df.loc[best_idx_accuracy]

#print best row
print("Best settings       *accuracy*      \n\n", best_row_accuracy,'\n')

#get index of best row based on f1 score 
best_idx_f1 = results_df['f1'].idxmax()
best_row_f1 = results_df.loc[best_idx_f1]

#print best row
print("Best settings       *f1*       \n\n", best_row_f1)

In [ ]:
# for each (FS_method, fs_param), find the index of the max accuracy
idx = (
    results_df
    .groupby(['FS_method', 'fs_param'])['accuracy']
    .idxmax()
)

# select those rows and reset the index
best_per_method_and_param = results_df.loc[idx].reset_index(drop=True)

# sort if you like
best_per_method_and_param = best_per_method_and_param.sort_values(
    [ 'accuracy'], ascending=[ False]
).reset_index(drop=True)

print(best_per_method_and_param)


In [ ]:
unique_count = len(results_df.drop_duplicates())
print(f"Number of unique rows (excluding index): {unique_count}")
